In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn imbalanced-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from imblearn.over_sampling import SMOTENC

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('Libraries loaded.')

Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing (a3 original) ─────────────────────────────────────
def preprocess(df):
    df = df.copy()
    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)
    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    # Fill NaN with -1 (needed for SMOTENC which requires no NaN)
    df = df.fillna(-1)
    return df


X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Features: {list(X_train.columns)}')

X_train: (13249, 59), X_test: (8834, 59)
Features: ['age', 'gender', 'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect', 'blood_cell_count', 'mother_age', 'father_age', 'alive', 'respiration', 'heart_rate', 'risk_level', 'birth_asphyxia', 'autopsy', 'place_birth', 'folic_acid', 'maternal_illness', 'radiation_exposure', 'substance_abuse', 'infertility_treatment', 'problem_previous_pregnancies', 'abortion_cnt', 'birth_defects', 'white_blood_cell_count', 'blood_test', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5', 'missing_count', 'missing_parent_age', 'missing_symptoms', 'missing_clinical', 'mother_age_missing', 'father_age_missing', 'maternal_defect_missing', 'gender_missing', 'risk_level_missing', 'heart_rate_missing', 'respiration_missing', 'abortion_cnt_missing', 'white_blood_cell_count_missing', 'defect_sum', 'symptom_sum', 'defect_x_symptom', 'any_defect', 'any_symptom', 'high_symptom', 'all_defects', 'parent_age_gap', 'symptom_defect_ratio', '

In [5]:
# ── Cell 5: Identify categorical feature indices for SMOTENC ─────────────────
# SMOTENC needs to know which cols are categorical so it interpolates them
# correctly (majority vote among neighbors) vs numeric (weighted average)

# After a3 preprocessing, these are the categorical columns (low cardinality integers)
cat_col_names = [
    'gender', 'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect',
    'alive', 'respiration', 'heart_rate', 'risk_level',
    'birth_asphyxia', 'place_birth', 'folic_acid', 'maternal_illness',
    'radiation_exposure', 'substance_abuse', 'infertility_treatment',
    'problem_previous_pregnancies', 'birth_defects', 'blood_test', 'autopsy',
    'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5',
    # engineered binary cols
    'any_defect', 'any_symptom', 'high_symptom', 'all_defects',
    's4_and_s5', 'no_s4_s5', 'both_parents_defect', 'no_parent_defect',
    # missing flag cols
    'mother_age_missing', 'father_age_missing', 'maternal_defect_missing',
    'gender_missing', 'risk_level_missing', 'heart_rate_missing',
    'respiration_missing', 'abortion_cnt_missing', 'white_blood_cell_count_missing',
]

# Get integer positions of cat cols in X_train
col_list = list(X_train.columns)
cat_indices = [col_list.index(c) for c in cat_col_names if c in col_list]
print(f'Categorical feature count for SMOTENC: {len(cat_indices)}')
print(f'Total features: {len(col_list)}')

Categorical feature count for SMOTENC: 42
Total features: 59


In [6]:
# ── Cell 6: SMOTENC — oversample only classes 0, 4, 8 ───────────────────────
# Strategy:
#   Class 4 (암):     58 → 300  (genuinely tiny, most impactful)
#   Class 8 (알츠하이머): 91 → 300  (second smallest)
#   Class 0 (레베르시):  389 → 500  (small, modest boost)
#   All others: leave exactly as-is
#
# We do NOT oversample class 9 — its problem is feature overlap, not size

TARGET_COUNTS = {
    0: 500,   # 레베르시: 389 → 500
    4: 300,   # 암: 58 → 300
    8: 300,   # 알츠하이머: 91 → 300
}

print('Oversampling strategy:')
class_counts = np.bincount(y_train)
for cls in range(10):
    n = class_counts[cls]
    target = TARGET_COUNTS.get(cls, n)
    action = f'→ {target}' if cls in TARGET_COUNTS else '(unchanged)'
    print(f'  Class {cls}: {n:4d} {action}')

# Build sampling_strategy dict — only specify classes to oversample
sampling_strategy = {cls: count for cls, count in TARGET_COUNTS.items()}

smote = SMOTENC(
    categorical_features = cat_indices,
    sampling_strategy    = sampling_strategy,
    k_neighbors          = 5,
    random_state         = 42,
)

print('\nRunning SMOTENC...')
X_train_sm, y_train_sm = smote.fit_resample(X_train.values, y_train)
X_train_sm = pd.DataFrame(X_train_sm, columns=X_train.columns)

print('\nClass distribution AFTER oversampling:')
vals, cnts = np.unique(y_train_sm, return_counts=True)
for v, c in zip(vals, cnts):
    old = class_counts[v]
    flag = f' (+{c-old})' if c > old else ''
    print(f'  Class {v}: {c}{flag}')
print(f'\nTotal rows: {len(X_train_sm)} (was {len(X_train)})')

Oversampling strategy:
  Class 0:  389 → 500
  Class 1: 2068 (unchanged)
  Class 2: 1090 (unchanged)
  Class 3: 3096 (unchanged)
  Class 4:   58 → 300
  Class 5: 1700 (unchanged)
  Class 6:  813 (unchanged)
  Class 7: 2643 (unchanged)
  Class 8:   91 → 300
  Class 9: 1301 (unchanged)

Running SMOTENC...

Class distribution AFTER oversampling:
  Class 0: 500 (+111)
  Class 1: 2068
  Class 2: 1090
  Class 3: 3096
  Class 4: 300 (+242)
  Class 5: 1700
  Class 6: 813
  Class 7: 2643
  Class 8: 300 (+209)
  Class 9: 1301

Total rows: 13811 (was 13249)


In [7]:
# ── Cell 7: Class weights on augmented dataset ────────────────────────────────
sm_class_counts  = np.bincount(y_train_sm, minlength=10)
sm_class_weights = len(y_train_sm) / (10 * sm_class_counts)

print('Class weights after SMOTENC:')
for i, (n, w) in enumerate(zip(sm_class_counts, sm_class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights after SMOTENC:
  Class 0: weight=2.762  (n=500)
  Class 1: weight=0.668  (n=2068)
  Class 2: weight=1.267  (n=1090)
  Class 3: weight=0.446  (n=3096)
  Class 4: weight=4.604  (n=300)
  Class 5: weight=0.812  (n=1700)
  Class 6: weight=1.699  (n=813)
  Class 7: weight=0.523  (n=2643)
  Class 8: weight=4.604  (n=300)
  Class 9: weight=1.062  (n=1301)


In [8]:
# ── Cell 8: Train CatBoost on SMOTE-augmented data — 3 seeds x 10 folds ─────
# IMPORTANT: OOF validation uses ONLY original train rows (no synthetic rows)
# This gives a fair comparison with previous runs
# The stratified split is done on the ORIGINAL data indices
# Synthetic rows are only in training, never in validation

n_orig = len(X_train)   # number of original rows

all_oof_proba  = np.zeros((n_orig, 10))
all_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    # Split on original indices only
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_proba   = np.zeros((n_orig, 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        # Training: original fold rows + ALL synthetic rows (rows beyond n_orig)
        synthetic_idx = np.arange(n_orig, len(X_train_sm))
        tr_combined   = np.concatenate([tr_idx, synthetic_idx])

        X_tr  = X_train_sm.iloc[tr_combined]
        y_tr  = y_train_sm[tr_combined]

        # Validation: original rows only — fair comparison
        X_val = X_train.iloc[val_idx]
        y_val = y_train[val_idx]

        model = CatBoostClassifier(
            iterations            = 2000,
            learning_rate         = 0.03,
            depth                 = 6,
            l2_leaf_reg           = 3,
            class_weights         = sm_class_weights,
            early_stopping_rounds = 100,
            eval_metric           = 'Accuracy',
            random_seed           = SEED,
            verbose               = 0,
            thread_count          = -1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    all_oof_proba  += oof_proba  / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

final_oof = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'SMOTENC OOF BA:      {final_oof:.4f}')
print(f'Best run OOF BA:     0.3933')
print(f'Change:              {final_oof - 0.3933:+.4f}')
print(f"{'='*60}")


======================================== SEED=42 ========================================
  Fold  1: BA=0.3893  best_iter=83
  Fold  2: BA=0.4117  best_iter=80
  Fold  3: BA=0.4315  best_iter=210
  Fold  4: BA=0.4039  best_iter=528
  Fold  5: BA=0.4089  best_iter=103
  Fold  6: BA=0.4567  best_iter=188
  Fold  7: BA=0.3627  best_iter=32
  Fold  8: BA=0.3702  best_iter=560
  Fold  9: BA=0.4277  best_iter=202
  Fold 10: BA=0.4255  best_iter=328
  OOF BA (seed=42): 0.4091 | mean=0.4088 ± 0.0273

======================================== SEED=7 ========================================
  Fold  1: BA=0.4246  best_iter=640
  Fold  2: BA=0.4399  best_iter=97
  Fold  3: BA=0.4095  best_iter=159
  Fold  4: BA=0.4145  best_iter=222
  Fold  5: BA=0.3709  best_iter=124
  Fold  6: BA=0.4338  best_iter=425
  Fold  7: BA=0.3964  best_iter=26
  Fold  8: BA=0.3820  best_iter=306
  Fold  9: BA=0.3976  best_iter=215
  Fold 10: BA=0.3995  best_iter=102
  OOF BA (seed=7): 0.4070 | mean=0.4069 ± 0.0209

====

In [9]:
# ── Cell 9: Per-class recall comparison ──────────────────────────────────────
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
best_recall = {0:0.314, 1:0.390, 2:0.270, 3:0.348, 4:0.793,
               5:0.339, 6:0.498, 7:0.238, 8:0.604, 9:0.139}

oof_labels = np.argmax(all_oof_proba, axis=1)
report     = classification_report(y_train, oof_labels, output_dict=True)

print(f'OOF BA: {final_oof:.4f}  (best LB run: 0.3933 OOF → 0.37127 LB)\n')
print(f'{"Class":<5} {"Name":<16} {"Best run":>10} {"Now":>8} {"Δ":>7} {"Smoted?":>8}')
print('-' * 60)
for cls in range(10):
    r      = report[str(cls)]['recall']
    r_old  = best_recall[cls]
    delta  = r - r_old
    smoted = '← SMOTED' if cls in TARGET_COUNTS else ''
    flag   = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
    print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>10.3f} {r:>8.3f} {delta:>+7.3f}  {smoted}{flag}')

OOF BA: 0.3985  (best LB run: 0.3933 OOF → 0.37127 LB)

Class Name               Best run      Now       Δ  Smoted?
------------------------------------------------------------
0     레베르시                  0.314    0.414  +0.100  ← SMOTED ← up
1     낭포성섬유증                0.390    0.385  -0.005  
2     당뇨                    0.270    0.327  +0.057   ← up
3     리증후군                  0.348    0.316  -0.032  
4     암                     0.793    0.655  -0.138  ← SMOTED
5     테이-삭스                 0.339    0.301  -0.038  
6     혈색소침착증                0.498    0.603  +0.105   ← up
7     사립체근병종                0.238    0.238  +0.000   ← LOW
8     알츠하이머                 0.604    0.560  -0.044  ← SMOTED
9     확인안됨                  0.139    0.185  +0.046   ← up


In [10]:
# ── Cell 10: Save submission ──────────────────────────────────────────────────
final_preds = np.argmax(all_test_preds, axis=1)
submission  = pd.DataFrame({
    'id'      : TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')
submission.to_csv('submission_smotenc.csv')

print('Saved: submission_smotenc.csv')
print(f'Shape: {submission.shape}')
print('\nPrediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print(f'\nFinal OOF BA: {final_oof:.4f}')
print(f'Submit only if OOF > 0.3933  (current best LB = 0.37127)')
print(f'Expected LB ≈ OOF − 0.022 based on gap pattern')

Saved: submission_smotenc.csv
Shape: (8834, 1)

Prediction distribution:
disorder
0     436
1    1350
2     727
3    1441
4     123
5    1206
6    1190
7    1080
8     134
9    1147
Name: count, dtype: int64

Final OOF BA: 0.3985
Submit only if OOF > 0.3933  (current best LB = 0.37127)
Expected LB ≈ OOF − 0.022 based on gap pattern
